# 02 · 因子回测入门：动量因子与前视偏差

**学习目标**：
1. 搭一个最简单的截面动量策略：每月买入过去 12 个月涨得最多的一组资产；
2. 亲眼看到量化新手最常犯的错误，**前视偏差（look-ahead bias）**，能让一个毫无意义的策略回测出惊人收益。

**数据**：50 个资产、6 年的**纯随机游走**价格（固定随机种子）。随机数据里没有任何可预测的规律，所以**正确的**回测结果应该接近“没有超额收益”。这正是检验回测代码是否写对的好办法。

> 本仓库仅供学习，所有内容不构成投资建议。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 Notebook 能直接复用 scripts/ 里的函数
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
def perf_stats(daily_ret: pd.Series, periods: int = 252) -> pd.Series:
    """年化收益、年化波动、夏普（无风险利率按 0）、最大回撤。"""
    daily_ret = daily_ret.dropna()
    nav = (1 + daily_ret).cumprod()
    ann_ret = nav.iloc[-1] ** (periods / len(daily_ret)) - 1
    ann_vol = daily_ret.std() * np.sqrt(periods)
    max_dd = (nav / nav.cummax() - 1).min()
    return pd.Series({"ann_return": ann_ret, "ann_vol": ann_vol,
                      "sharpe": ann_ret / ann_vol if ann_vol else np.nan, "max_drawdown": max_dd})

## 1. 生成随机价格

每个资产的日收益独立同分布，均值 0.03%、日波动 2%。

In [ ]:
rng = np.random.default_rng(2026)
n_assets, n_days = 50, 252 * 6
dates = pd.bdate_range("2020-01-01", periods=n_days)
daily = pd.DataFrame(rng.normal(0.0003, 0.02, (n_days, n_assets)), index=dates,
                     columns=[f"A{i:02d}" for i in range(n_assets)])
prices = (1 + daily).cumprod()
monthly = prices.resample("ME").last()
mret = monthly.pct_change()
mret.shape

## 2. 动量信号

信号 = 过去 12 个月的累计收益。每月末按信号排序，买入前 20%（10 个资产）等权持有一个月，基准是全部 50 个资产等权。

关键问题：**月末算出的信号，能不能用来赚“同一个月”的收益？**

In [ ]:
signal = monthly.pct_change(12)          # 截至当月末的过去 12 个月收益

def backtest(signal_used: pd.DataFrame, top_frac: float = 0.2) -> pd.Series:
    """每月按 signal_used 选前 top_frac 的资产，等权持有，返回月度收益。"""
    k = int(signal_used.shape[1] * top_frac)
    picks = signal_used.rank(axis=1, ascending=False) <= k
    port = (mret * picks).sum(axis=1) / picks.sum(axis=1)
    return port[picks.sum(axis=1) == k]

benchmark = mret.mean(axis=1)

## 3. 错误写法：用当月信号赚当月收益（前视偏差）

`signal` 在第 t 个月末才算得出来，里面**已经包含了第 t 个月的收益**。拿它去选第 t 个月的持仓，等于提前知道了答案。

In [ ]:
wrong = backtest(signal)                  # ❌ 没有滞后
right = backtest(signal.shift(1))         # ✅ 用上个月末的信号决定本月持仓
bench = benchmark.loc[right.index]

def monthly_stats(r):
    return perf_stats(r, periods=12)

pd.DataFrame({"look-ahead (wrong)": monthly_stats(wrong.loc[right.index]),
              "lagged (right)": monthly_stats(right),
              "equal-weight benchmark": monthly_stats(bench)})

In [ ]:
nav = pd.DataFrame({"look-ahead (wrong)": (1 + wrong.loc[right.index]).cumprod(),
                    "lagged (right)": (1 + right).cumprod(),
                    "benchmark": (1 + bench).cumprod()})
ax = nav.plot(figsize=(9, 4), logy=True, title="Momentum on pure random walks")
ax.set_ylabel("NAV (log scale)")
plt.show()

## 4. 怎么读这个结果

- **错误写法**在纯随机数据上也能跑出很高的年化收益和夏普，因为它“偷看”了要预测的那个月。
- **正确写法**（信号滞后一期）没有任何超额收益，甚至跑输等权基准：随机数据本来就没有动量，而只持有 10 个资产，分散度比基准低、波动更高。
- 结论：**看到漂亮的回测，第一件事是检查信号和收益在时间上有没有错位**，其次才是检查交易成本、幸存者偏差和过拟合。

## 练习

1. 加入交易成本：每月换手部分按单边 0.1% 扣费，`right` 的收益变成多少？
2. 把随机数据换成带有自相关的序列（例如让每个资产的日收益带一个持续的漂移项），动量策略还能赚钱吗？
3. 用 [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) 的动量因子（Mom）数据，看真实市场中动量因子的长期表现和大幅回撤（例如 2009 年）。